In [ ]:
!pip -q install numpy pandas scikit-learn faker tqdm joblib pydantic flask

import os, json
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
FEATURE_COLS = [
    "wordCount","skillCount","claimedYears","hasGithub","hasLinks","educationMentions",
    "certMentions","projectCount","avgProjectAgeMonths","timelineGapMonths",
    "contributionCount90d","contributionStreakWeeks","institutionRank","skillConsistency",
]

In [ ]:
def _clip(x, lo, hi):
    return float(max(lo, min(hi, x)))

def generate_one(rng):
    skill_depth = rng.beta(2.2, 2.2)
    project_quality = rng.beta(2.0, 2.2)
    discipline = rng.beta(2.2, 1.8)
    honesty = rng.beta(3.0, 1.4)
    institution_rank = int(rng.integers(1, 6))
    fraud = 1 if rng.random() > honesty else 0

    word_count = _clip(rng.normal(450 + 500 * skill_depth, 180), 80, 1800)

    true_skill_count = int(_clip(rng.normal(5 + 10 * skill_depth, 2.5), 0, 35))
    skill_inflation = rng.uniform(0.15, 1.0) if fraud else 0.0
    skill_count = int(_clip(true_skill_count * (1.0 + 0.7 * skill_inflation), 0, 50))

    years_true = _clip(rng.normal(1.0 + 10.0 * skill_depth, 2.0), 0, 18)
    claimed_years = _clip(years_true * (1.0 + (0.4 + skill_inflation) if fraud else 1.0), 0, 25)

    project_count_true = int(_clip(rng.poisson(1 + 7 * project_quality), 0, 30))
    project_count = int(_clip(project_count_true * (1.0 + (0.5 * skill_inflation if fraud else 0.0)), 0, 50))

    avg_project_age_months = _clip(rng.normal(8 + 30 * project_quality + 10 * skill_depth, 10), 0, 240)
    if fraud:
        avg_project_age_months = _clip(avg_project_age_months - rng.uniform(5, 20), 0, 240)

    gap_base = _clip(rng.normal(1 + 10 * (1 - discipline), 5), 0, 60)
    timeline_gap_months = gap_base
    if fraud:
        timeline_gap_months = _clip(timeline_gap_months + rng.uniform(6, 30), 0, 120)

    has_links = 1 if rng.random() < _clip(0.35 + 0.4 * skill_depth + 0.2 * project_quality, 0.0, 1.0) else 0
    has_github = 1 if rng.random() < _clip(0.25 + 0.55 * skill_depth, 0.0, 1.0) else 0
    if fraud and rng.random() < 0.2:
        has_links = 1
        has_github = 1

    education_mentions = int(_clip(rng.poisson(1 + (6 - institution_rank) * 0.25), 0, 10))
    cert_mentions = int(_clip(rng.poisson(0.6 + 1.8 * skill_depth), 0, 20))
    if fraud:
        cert_mentions = int(_clip(cert_mentions + rng.integers(0, 3), 0, 20))

    contribution_count90d = int(_clip(rng.poisson(5 + 80 * skill_depth + 30 * project_quality), 0, 5000))
    contribution_streak_weeks = int(_clip(rng.poisson(1 + 10 * skill_depth), 0, 520))
    if has_github == 0:
        contribution_count90d = int(_clip(contribution_count90d * 0.2, 0, 5000))
        contribution_streak_weeks = int(_clip(contribution_streak_weeks * 0.3, 0, 520))

    evidence = np.tanh((contribution_count90d / 60.0) + (project_count / 8.0) + (claimed_years / 8.0))
    claimed_pressure = np.tanh(skill_count / 12.0)
    skill_consistency = _clip(0.15 + 0.75 * evidence - 0.35 * claimed_pressure + 0.25 * skill_depth, 0.0, 1.0)
    if fraud:
        skill_consistency = _clip(skill_consistency - rng.uniform(0.05, 0.35), 0.0, 1.0)

    inst_score = (6 - institution_rank) / 5.0

    score_base = (
        20 * np.tanh(word_count / 700.0) +
        18 * (skill_depth) +
        16 * (project_quality) +
        10 * inst_score +
        12 * np.tanh(contribution_count90d / 120.0) +
        6 * np.tanh(contribution_streak_weeks / 10.0) +
        10 * skill_consistency +
        6 * (1.0 - np.tanh(timeline_gap_months / 18.0)) +
        2 * has_github +
        0.5 * has_links
    )
    score_base = float(_clip(score_base, 0, 100))

    fraud_penalty = 18.0 if fraud else 0.0
    incons_penalty = 12.0 * (1.0 - skill_consistency) + 8.0 * np.tanh(timeline_gap_months / 24.0)

    label_score = float(_clip(score_base - fraud_penalty - incons_penalty, 0, 100))
    label_fraud = int(fraud)

    row = {
        "wordCount": float(word_count),
        "skillCount": float(skill_count),
        "claimedYears": float(claimed_years),
        "hasGithub": float(has_github),
        "hasLinks": float(has_links),
        "educationMentions": float(education_mentions),
        "certMentions": float(cert_mentions),
        "projectCount": float(project_count),
        "avgProjectAgeMonths": float(avg_project_age_months),
        "timelineGapMonths": float(timeline_gap_months),
        "contributionCount90d": float(contribution_count90d),
        "contributionStreakWeeks": float(contribution_streak_weeks),
        "institutionRank": float(institution_rank),
        "skillConsistency": float(skill_consistency),
        "label_score": label_score,
        "label_fraud": label_fraud,
    }
    return row

def make_dataset(n=20000, seed=42):
    rng = np.random.default_rng(seed)
    rows = [generate_one(rng) for _ in tqdm(range(n))]
    return pd.DataFrame(rows)

df = make_dataset(20000, 42)
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score, average_precision_score, brier_score_loss
import joblib

X = df[FEATURE_COLS].astype(float)
y_score = df["label_score"].astype(float).values
y_fraud = df["label_fraud"].astype(int).values

X_train, X_test, y_score_train, y_score_test, y_fraud_train, y_fraud_test = train_test_split(
    X, y_score, y_fraud, test_size=0.2, random_state=42, stratify=y_fraud
)

score_model = GradientBoostingRegressor(random_state=42, n_estimators=350, learning_rate=0.05, max_depth=3, subsample=0.9)
fraud_model = GradientBoostingClassifier(random_state=42, n_estimators=350, learning_rate=0.05, max_depth=3, subsample=0.9)

score_model.fit(X_train, y_score_train)
fraud_model.fit(X_train, y_fraud_train)

pred_score = score_model.predict(X_test)
fraud_proba = fraud_model.predict_proba(X_test)[:,1]

metrics = {
    "score_model": {
        "mae": float(mean_absolute_error(y_score_test, pred_score)),
        "rmse": float(mean_squared_error(y_score_test, pred_score) ** 0.5),
        "r2": float(r2_score(y_score_test, pred_score))
    },
    "fraud_model": {
        "roc_auc": float(roc_auc_score(y_fraud_test, fraud_proba)),
        "avg_precision": float(average_precision_score(y_fraud_test, fraud_proba)),
        "brier": float(brier_score_loss(y_fraud_test, fraud_proba)),
        "fraud_rate_test": float(np.mean(y_fraud_test))
    }
}
metrics

In [ ]:
os.makedirs("artifacts", exist_ok=True)
joblib.dump(score_model, "artifacts/score_model.pkl")
joblib.dump(fraud_model, "artifacts/fraud_model.pkl")

with open("artifacts/feature_cols.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)

medians = X_train.median().to_dict()
with open("artifacts/feature_medians.json", "w") as f:
    json.dump({k: float(v) for k,v in medians.items()}, f, indent=2)

with open("artifacts/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# optional dataset export
os.makedirs("data", exist_ok=True)
df.to_csv("data/synthetic_resumes.csv", index=False)

print("Saved artifacts/ and data/")